# 02 — Veri hazırlama ve özellik mühendisliği

Bu notebook tek başına çalışır; `01_eda.ipynb` oturumundaki değişkenleri kullanmaz.
Gerekçeler [EDA 12. bölüm](01_eda.ipynb) ve [hazırlık planındadır](../docs/data_preparation_plan.md).
İlk uygulama URL-only `url_lexical_v1` sözleşmesidir. Eski 56 sütunlu veri
yerinde düzeltilmez; yeni özellikler ortak modülle yeniden üretilir.

**Sıra:** Bulgu → Karar → Uygulama → Öncesi/sonrası doğrulama.
Bu süreç model eğitmez, API değiştirmez ve URL ziyaret etmez.
Doğrulama/testte yalnız önceden tanımlı deterministik dönüşüm ve bütünlük
kontrolü yapılır; dağılımlar veya performans üzerinden karar üretilmez.

## 1. Kaynak, ayrım ve ortam

EDA'daki kaynak ve host ayrımı korunur; eksik manifesti otomatik yeniden
oluşturmayız. Farklı kaynak/ayar tespitinde işlem durur. Python 3.12 ve
`requirements-eda.txt` ortamını kullanın. Çekirdeği kod değişikliğinden sonra
yeniden başlatıp tüm hücreleri sırayla çalıştırın.

In [1]:
from pathlib import Path
import sys, json, hashlib, platform
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/data/preparation.py').is_file())
sys.path.insert(0,str(ROOT))
from src.features import FEATURE_NAMES, SCHEMA_VERSION, extract_url_features, hostname_group
from src.data.preparation import prepare_project
lock=json.loads((ROOT/'data/processed/split_metadata.json').read_text(encoding='utf-8'))
assert hashlib.sha256((ROOT/'data/raw/PhiUSIIL_Phishing_URL_Dataset.csv').read_bytes()).hexdigest()==lock['source_sha256']
print('Python:',platform.python_version())
print('Source SHA-256:',lock['source_sha256'])
print('Split policy:',lock['group_rule'])

Python: 3.12.10
Source SHA-256: a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6
Split policy: parsed_hostname_idna_lower_strip_trailing_dot_v1


## 2. Açık özellik sözleşmesi ve farkın çözümü

**Bulgu (EDA 6.2, 10 ve 11):** Kayıtlı uzunluk/HTTPS/sayaç değerlerini tam
yeniden üretemiyoruz. **Karar:** Kayıtlı sütunları sessizce değiştirmek yerine
yeni alanları tek bir offline fonksiyonla tanımlamak.

`has_https` yalnız parser şemasından gelir; query içindeki https etkisizdir.
`url_length` özgün metnin Unicode karakter sayısıdır. Harf büyüklüğü,
percent-encoding ve query sırası URL eşitliği için değiştirilmez.
Hostname IDNA normalizasyonu gruplama içindir; `hostname_length` parsed
hostname'in küçük harf biçimini ölçer. TLD/subdomain sayısı, referans skorları,
HTML, sertifika geçerliliği ve domain yaşı bu sürümde yoktur.
Geçersiz girdiye sahte sıfır özellik yerine açıklamalı hata üretilir.

In [2]:
example='http://user@example.com:8080/a-1?next=https://other.com&x=%20'
example_features=extract_url_features(example)
assert tuple(example_features)==FEATURE_NAMES
assert example_features['has_https']==0
display(pd.Series(example_features).to_frame('Value'))
print('Observed hostname group:',hostname_group(example))
print('Şema sürümü:',SCHEMA_VERSION)

,Value
url_length,61.000000
hostname_length,11.000000
has_at_symbol,1.000000
has_dash,1.000000
count_dots,2.000000
has_https,0.000000
digit_count,7.000000
digit_ratio,0.114754
path_length,4.000000
query_length,28.000000


Observed hostname group: example.com
Şema sürümü: url_lexical_v1


## 3. Uygulama: temsilci kayıt, hedef ve çıktı dosyaları

**Bulgu (EDA 5):** Aynı exact URL'deki kayıtlı özellikler farklı; etiketler aynı.
**Karar:** Eski ölçümleri keyfi seçmek yerine URL-only yeni özellikleri ortak
fonksiyonla üret. Aynı URL ve hedef için tek lexical gözlem yeterlidir;
en küçük kaynak satırını temsilci tut, bütün kaynak satırlarını eşlemede koru.
Bu, eski içerik gözlemlerinin doğruluğu hakkında karar değildir.

Çelişkili etiket, aynı URL/host'un farklı partition'larda olması veya eksik
manifest işlemi durdurur. Geçersiz URL'ler sebebiyle karantinaya alınır.
Öğrenilen doldurma/ölçekleme ve IQR silme/kırpma uygulanmaz.

**Dosyalar:** `data/processed/url_lexical_v1/` içinde `features.csv`, `target.csv`,
`metadata.csv`, `source_mapping.csv`, `quarantine.csv`, `schema.json`.
CSV'lerdeki `source_row` yalnız birleşim anahtarıdır; model girdisi değildir.
Hedef `is_phishing=1` phishing, `0` meşrudur. Partition metadata'dadır.

In [3]:
result=prepare_project(ROOT)
X=result['tables']['X']
y=result['tables']['y']
metadata=result['tables']['metadata']
mapping=result['tables']['mapping']
quarantine=result['tables']['quarantine']
display(pd.Series({k:result['summary'][k] for k in ['source_rows','output_rows','duplicate_rows_mapped','quarantined_rows','schema','target']}).to_frame('Value'))
display(result['partition_summary'])
display(X.head())
display(mapping.head())
display(quarantine)
assert tuple(X.columns)==FEATURE_NAMES and 'label' not in X and 'is_phishing' not in X
assert X.index.equals(y.index) and X.index.equals(metadata.index)
assert np.isfinite(X.to_numpy()).all()
print('Kimlik, URL, hostname ve partition X içinde yok.')

,Value
source_rows,235795
output_rows,235370
duplicate_rows_mapped,425
quarantined_rows,0
schema,url_lexical_v1
target,"is_phishing: 1 phishing, 0 legitimate"


,rows,host_groups
partition,,
test,47514,44016
train,141001,132045
validation,46855,44016


,url_length,hostname_length,has_at_symbol,has_dash,count_dots,has_https,digit_count,digit_ratio,path_length,query_length,count_query_parameters,count_percent_escapes,hostname_is_ip,has_explicit_port
source_row,,,,,,,,,,,,,,
0,32,24,0,0,2,1,0,0.0,0,0,0,0,0,0
1,24,16,0,1,2,1,0,0.0,0,0,0,0,0,0
2,30,22,0,0,3,1,0,0.0,0,0,0,0,0,0
3,27,19,0,0,2,1,0,0.0,0,0,0,0,0,0
4,34,26,0,0,2,1,0,0.0,0,0,0,0,0,0


,source_row,partition,representative_source_row,status
0,0,train,0,representative
1,1,train,1,representative
2,2,train,2,representative
3,3,test,3,representative
4,4,train,4,representative


,source_row,reason


Kimlik, URL, hostname ve partition X içinde yok.


## 4. Çıktıyı diskten okuyup doğrulama

Eşitlik yalnız bellekte değil teslim dosyalarında da kontrol edilir. Kaynak satır
anahtarları, hedef dönüşümü, kalan URL benzersizliği, partition bağlantısı ve
host örtüşmesi doğrulanır. Az sayıda sabit satırda eğitim/canlı extractor parity
kontrolü yapılır; bağımsız sınır durumları `tests/` altındadır.

In [4]:
OUTPUT=ROOT/'data/processed'/SCHEMA_VERSION
disk_X=pd.read_csv(OUTPUT/'features.csv',index_col='source_row')
disk_y=pd.read_csv(OUTPUT/'target.csv',index_col='source_row')
disk_meta=pd.read_csv(OUTPUT/'metadata.csv',index_col='source_row')
pd.testing.assert_frame_equal(disk_X,X,check_exact=False,rtol=1e-12,atol=1e-12)
pd.testing.assert_frame_equal(disk_y,y)
pd.testing.assert_frame_equal(disk_meta,metadata)
assert not disk_meta.URL.duplicated().any()
assert disk_meta.groupby('hostname_group').partition.nunique().max()==1
assert np.array_equal(disk_y.is_phishing.to_numpy(),1-disk_meta.original_label.to_numpy())
for row in disk_X.index[np.linspace(0,len(disk_X)-1,min(12,len(disk_X)),dtype=int)]:
    expected=extract_url_features(disk_meta.loc[row,'URL'])
    np.testing.assert_allclose(disk_X.loc[row,list(FEATURE_NAMES)],list(expected.values()),rtol=1e-12,atol=1e-12)
assert len(mapping)==result['summary']['source_rows']
assert set(mapping.loc[mapping.status.ne('quarantined'),'representative_source_row'])==set(X.index)
print('PASS: disk outputs, X/y alignment, target, host separation, lineage and extractor parity.')
display(Markdown(f"**Sonuç:** {result['summary']['source_rows']:,} kaynak satırından {result['summary']['output_rows']:,} lexical gözlem üretildi; {result['summary']['duplicate_rows_mapped']:,} tekrar satırı temsilciye eşlendi, {result['summary']['quarantined_rows']:,} satır karantinada. İşlem sonucu eğitim/validasyon/test satır sayıları yukarıda raporlandı; yeni ayrım yapılmadı."))

PASS: disk outputs, X/y alignment, target, host separation, lineage and extractor parity.


**Sonuç:** 235,795 kaynak satırından 235,370 lexical gözlem üretildi; 425 tekrar satırı temsilciye eşlendi, 0 satır karantinada. İşlem sonucu eğitim/validasyon/test satır sayıları yukarıda raporlandı; yeni ayrım yapılmadı.

## 5. Teslim ve açık kalan işler

Rol 2 `schema.json` içindeki sıralı `features` listesini X olarak seçmeli;
`source_row` veya metadata sütunlarını modele vermemeli. `target.csv` ile
`metadata.csv` dosyaları anahtar üzerinden birleştirilir. Ön işleme yalnız
eğitimde fit edilir. Kaynak/çıktı/modül hash'leri ve sürümler
`reports/data_preparation/summary.json` içindedir.

Ortak çıkarıcı uygulanmıştır; **backend entegrasyonu ve model eğitimi yapılmamıştır.**
Mevcut API'nin eski extractor'ı/heuristic fallback'i bu sözleşmeyle uyumlu değildir.
Host ayrımı registered-domain veya kampanya bağımsızlığını kanıtlamaz; önceki
çalışmalarda görülen veri dış test sayılmaz. Sertifika ve domain yaşı ek işlerdir.

Tekrar üretim: Bu notebook'ta Run All. Bu işlem yalnız sürümlenmiş yerel çıktı
dosyalarını ve hazırlık raporunu yeniler; ham veri, EDA ve API korunur.
HTML görünümü ayrıca export edilir; Run All HTML'yi kendiliğinden yenilemez.

## 6. EDA bulguları için öncesi/sonrası ve durum takibi

**Amaç:** “Yeni çıktıda doğrulandı”, “işlem gerekmiyor”, “kapsam dışında
bırakıldı” ve “hâlâ açık” durumlarını birbirine karıştırmamak.

**Kapsam:** Öncesi/sonrası incelemesi EDA ile aynı **141.254 kaynak eğitim
satırında** yapılır. Tekilleşen kayıtlar temsilci eşlemesiyle yeni değerlere
bağlanır; böylece örneklem değiştiği için daha az fark varmış gibi görünmez.
Tüm kümelerde yalnız şema/anahtar/target/grup bütünlüğü kontrol edilir.

**Uygulanan düzeltmenin anlamı:** `url_length`, `has_https`, `hostname_group`,
`digit_count` ve `digit_ratio` açık yeni tanımlarla doğrulanır. Eski 56 sütun
yerinde değiştirilmez ve orijinal üretici formülünün bulunduğu iddia edilmez.
Eski değer farklıysa mutlaka eski veri yanlış demeyiz; yeni sözleşmeye göre
doğru çıktıyı teslim eder ve eski tanım sorusunu açık bırakırız.

`=`, `?`, `&` sayıları da eğitimde doğrudan yeniden hesaplanır; fark kayıtları
yerel inceleme dosyasına yazılır. **Bu üç tanısal alan 14 özellikli X'e eklenmez.**
Her tanısal hesap, otomatik yeni model özelliği değildir.

Kaynak satırı bazlı farklar `data/processed/url_lexical_v1/` altında yereldir;
küçük özetler `reports/data_preparation/` altında saklanır. Ham URL'ler
karşılaştırma raporuna yazılmaz. Hiçbir mevcut veri dosyası yeniden düzeltilmez.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/data/remediation_audit.py').is_file())
sys.path.insert(0,str(ROOT))
from src.data.remediation_audit import audit_preparation
remediation=audit_preparation(ROOT)
display(remediation['comparisons'])
display(pd.Series(remediation['domain']).to_frame('Value'))
status_names={
    'applied_verified':'Uygulandı / doğrulandı',
    'applied_verified_new_contract':'Yeni tanım uygulandı / doğrulandı',
    'no_action_needed':'Bu veride işlem gerekmiyor',
    'diagnostic_only_not_selected':'Tanısal hesaplandı / X alanı değil',
    'excluded_not_resolved':'Kapsam dışında / sorun çözülmedi',
    'no_automatic_edit':'Otomatik düzeltme gerekçesi yok',
    'not_needed_at_this_stage':'Bu aşamada uygulanmaz',
    'open':'Açık',
    'not_implemented':'Uygulanmadı'}
view=remediation['ledger'].copy()
view['durum']=view.status.map(status_names)
with pd.option_context('display.max_rows',None,'display.max_colwidth',100):
    display(view[['id','eda_finding','durum','action_or_evidence','remaining_limit']])
assert remediation['comparisons'].new_definition_mismatches.eq(0).all()
assert remediation['domain']['new_hostname_group_mismatches']==0
print('PASS: declared new definitions, lineage and partition checks; raw/EDA/API unchanged.')


,recorded_field,prepared_field,training_source_rows,old_definition_mismatches,new_definition_mismatches,status
0,URLLength,url_length,141254,112331,0,new_contract_verified; original_formula_not_cl...
1,IsHTTPS,has_https,141254,271,0,new_contract_verified; original_formula_not_cl...
2,NoOfEqualsInURL,[audit only],141254,168,0,diagnostic_recomputed_only; not_a_model_feature
3,NoOfQMarkInURL,[audit only],141254,3,0,diagnostic_recomputed_only; not_a_model_feature
4,NoOfAmpersandInURL,[audit only],141254,1740,0,diagnostic_recomputed_only; not_a_model_feature
5,NoOfDegitsInURL,digit_count,141254,685,0,new_contract_verified; original_formula_not_cl...
6,DegitRatioInURL,digit_ratio,141254,29404,0,new_contract_verified; original_formula_not_cl...


,Value
mismatching_training_rows,18
port_cases,16
userinfo_cases,2
recorded_domain_equals_netloc_rows,141254
new_hostname_group_mismatches,0


,id,eda_finding,durum,action_or_evidence,remaining_limit
0,source,Raw data unchanged,Uygulandı / doğrulandı,"Source, split, EDA and backend hashes preserved",Original stays authoritative; no overwrite
1,quality,Missing/nonfinite values,Bu veride işlem gerekmiyor,No original missing/infinite cells; new X finite,"Future invalid input is rejected/quarantined, not imputed globally"
2,target,Opposite source/API target convention,Uygulandı / doğrulandı,New is_phishing=1; original_label kept outside X,Backend integration is a separate open item
3,duplicates,425 extra URL observations,Uygulandı / doğrulandı,235370 URL-only representatives; all source lineage retained,Old content observations were not adjudicated
4,split,Group overlap,Uygulandı / doğrulandı,Original partitions inherited; no observed-host overlap,Registered-domain/campaign independence remains open
5,url_length,Recorded URLLength mismatch,Yeni tanım uygulandı / doğrulandı,New url_length matches original text length,Why the original differs by -1 remains unknown
6,https,HTTPS indicator semantics,Yeni tanım uygulandı / doğrulandı,New has_https uses parsed scheme,Original producer code not verified; no TLS certificate check
7,domain,Netloc versus hostname,Yeni tanım uygulandı / doğrulandı,Separate parsed hostname_group; port/userinfo classified,Original Domain retained; no registered-domain extraction
8,digits,Digit count and ratio,Yeni tanım uygulandı / doğrulandı,ASCII digit_count and documented denominator verified,Original ratio formula not reconstructed
9,special_counts,Equals/question/ampersand mismatches,Tanısal hesaplandı / X alanı değil,Recomputed training counts and differences saved locally,These three counts are not in the 14-field model contract


PASS: declared new definitions, lineage and partition checks; raw/EDA/API unchanged.


## 7. Bu kontrolden sonra neyi çözülmüş sayıyoruz?

- **Yeni çıktı için doğrulananlar:** Açık uzunluk/HTTPS/digit tanımları,
  parser'dan ayrı hostname, ters etiket kodlaması, URL-only tekrar ağırlığı,
  kaynak satırı eşlemesi ve mevcut host ayrımının korunması.
- **Farkı açıklanan konu:** Kaynak Domain alanının hostname yerine port veya
  userinfo içeren netloc tutabilmesi. Eğitimdeki 18 farkın 16'sında port,
  2'sinde kullanıcı bilgisi vardı. Yeni hostname bu parçaları içermez.
- **Yeni tanısal çıktı:** `=`, `?`, `&` sayıları ve eski sayımlardan farklar;
  X'e eklenmiş özellik veya üretici kodunun doğrulaması değildir.
- **Düzeltme gerektirmeyen bulgular:** Eksik değer olmaması, seyrek özellik,
  uzun kuyruk ve yüksek korelasyon tek başına hata değildir. Bu nedenlerle
  veri silmiyoruz veya yapay değer üretmiyoruz.
- **Çözülmeyenler:** Orijinal oran/uzunluk/score formüllerinin kökeni, sayfa
  ölçümleri, referans skorlarının hesaplama evreni, TLD/suffix kuralları,
  backend entegrasyonu, dış/zamansal/kampanya değerlendirmesi ve ağ gerektiren
  domain yaşı/sertifika özellikleri. Çıktıdan çıkarılmış olmaları çözüm değildir.

**Sonuç:** EDA'daki her bulguya bir durum atanmıştır; “bütün 56 sütun
düzeltildi” iddiası yoktur. Yapılabilen deterministik veri hazırlama işlemleri
uygulanıp doğrulanmıştır. Kaynak kanıtı gerektiren konular açık kayıt olarak
korunur. Güncel takip: `reports/data_preparation/finding_status.csv`.
